# Session 6. Streaming and the human in the loop

**One agent, three new controls: watch it, stop it, rewind it.**

- three stream modes over one run: `values`, `updates`, `messages`
- `interrupt()` parks a run mid-tool; `Command(resume=...)` picks it up
- every step is a checkpoint you can replay, or fork

In [ ]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()  # reads .env once; nothing below opens a file


def chat_model(size: str = "cheap", **kwargs):
    """A model object for the configured provider. A dozen lines, copy them once."""
    name = os.environ[f"MODEL_{size.upper()}"]  # ids live in .env, never in code
    secret = os.environ["LLM_API_KEY"]
    if os.getenv("LLM_REASONING_EFFORT"):  # gpt-5.x: tools need reasoning "none"
        kwargs.setdefault("reasoning_effort", os.environ["LLM_REASONING_EFFORT"])
    # Gemini over OpenAI-compat drops the reasoning signature: turn two 400s
    if os.getenv("LLM_PROVIDER", "openai_compat") == "google_genai":
        return init_chat_model(f"google_genai:{name}", api_key=secret, **kwargs)
    return init_chat_model(
        f"openai:{name}", api_key=secret, base_url=os.environ["LLM_BASE_URL"], **kwargs
    )


print("provider:", os.getenv("LLM_PROVIDER", "openai_compat"),
      "| strong:", os.environ["MODEL_STRONG"])

## Three ways to watch a run

**`invoke` is opaque: the answer arrives all at once, at the end.**

- today's domain is email dispatch; `send_email` is a stub and `OUTBOX` is the only mailbox
- `stream` yields while the graph works; `stream_mode` picks what it yields
- first, a two-node drafting graph: compose with the model, polish in Python

In [ ]:
from langchain_core.messages import AIMessage
from langgraph.graph import END, START, MessagesState, StateGraph

model = chat_model("strong")  # one model object for the whole session

ASK = "Draft a short note telling the dean the quarterly report is ready. Body only: no subject line, no signature."


def compose(state: MessagesState) -> dict:
    return {"messages": [model.invoke(state["messages"])]}


def polish(state: MessagesState) -> dict:
    draft = state["messages"][-1].content  # pure Python: no model call inside
    note = f"Subject: Quarterly report\n\n{draft}\n\n-- The dispatch desk"
    return {"messages": [AIMessage(note)]}


builder = StateGraph(MessagesState)
builder.add_node("compose", compose)
builder.add_node("polish", polish)
builder.add_edge(START, "compose")
builder.add_edge("compose", "polish")
builder.add_edge("polish", END)
drafting = builder.compile()

result = drafting.invoke(
    {"messages": [{"role": "user", "content": ASK}]},
    config={"recursion_limit": 5},
)
print(result["messages"][-1].content)  # everything, at once, at the end

**`values` mode: the whole state after every step.**

- one chunk per step, and each chunk is the full state dict
- the right feed for a state inspector or a debugger view
- heavyweight: the whole history rides in every chunk

In [ ]:
for state in drafting.stream(
    {"messages": [{"role": "user", "content": ASK}]},
    config={"recursion_limit": 5},
    stream_mode="values",
):
    last = state["messages"][-1]  # the state so far, not a delta
    print(len(state["messages"]), "messages | last:", type(last).__name__)

**`updates` mode: only the delta, keyed by the node that made it.**

- one chunk per node run: `{node_name: partial_update}`
- the natural feed for progress lines: compose done, polish done
- compare with the previous cell: the delta is a fraction of the state

In [ ]:
for update in drafting.stream(
    {"messages": [{"role": "user", "content": ASK}]},
    config={"recursion_limit": 5},
    stream_mode="updates",
):
    for node, delta in update.items():  # one entry per node that just ran
        print(f"{node}: {delta['messages'][-1].content[:42]!r}")

**`messages` mode: model tokens as they are generated, plus metadata.**

- chunks of `(message_chunk, metadata)`; every chat UI eats this feed
- `metadata["langgraph_node"]` names the producer: filter on it
- the committed run replays whole replies; live, this trickles word by word

In [ ]:
for chunk, meta in drafting.stream(
    {"messages": [{"role": "user", "content": ASK}]},
    config={"recursion_limit": 5},
    stream_mode="messages",
):
    if meta["langgraph_node"] == "compose" and chunk.content:  # compose tokens only
        print(chunk.content, end="", flush=True)
print()

**Modes combine, and the docs grew a second surface.**

- `stream_mode=["updates", "messages"]` yields `(mode, chunk)` tuples
- further modes exist: `checkpoints`, `tasks`, `debug`, `custom`
- the docs now also show a typed `stream_events` v3 surface; the course stays on `stream_mode`, current and stable

## The dispatch agent

**The session-2 loop, unchanged, with one dangerous tool in it.**

- `lookup_contact` reads a fake address book: safe
- `send_email` appends to `OUTBOX`, the only mailbox today: nothing real is sent
- the builder becomes a function because today it compiles four times

In [ ]:
from langchain_core.tools import tool

# a fake address book: fictional people, no network
CONTACTS = {
    "dean": "dean@university.example",
    "lab manager": "lab-manager@university.example",
    "registrar": "registrar@university.example",
}
OUTBOX = []  # send_email writes here and nowhere else


@tool
def lookup_contact(name: str) -> str:
    """Look up a person's email address in the department address book."""
    key = name.strip().lower()
    if key not in CONTACTS:
        # a miss the model can act on, not an exception
        return f"No contact {name!r}. Known names: {', '.join(sorted(CONTACTS))}."
    return CONTACTS[key]


@tool
def send_email(to: str, body: str) -> str:
    """Send an email to an address obtained from lookup_contact."""
    OUTBOX.append({"to": to, "body": body})  # the irreversible act, unguarded
    return f"sent to {to}"


print(lookup_contact.invoke({"name": "dean"}))
print(lookup_contact.invoke({"name": "provost"}))

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition

SYSTEM = "You dispatch email. Find the address with lookup_contact before any send."
REQUEST = "Tell the dean the quarterly report is ready."


def dispatch_builder(toolset):
    bound = model.bind_tools(toolset)

    def call_model(state: MessagesState) -> dict:
        return {"messages": [bound.invoke(state["messages"])]}

    builder = StateGraph(MessagesState)
    builder.add_node("model", call_model)
    builder.add_node("tools", ToolNode(toolset))
    builder.add_edge(START, "model")
    builder.add_conditional_edges("model", tools_condition)
    builder.add_edge("tools", "model")
    return builder


def dispatch_input():
    return {"messages": [{"role": "system", "content": SYSTEM},
                         {"role": "user", "content": REQUEST}]}


open_builder = dispatch_builder([lookup_contact, send_email])
open_graph = open_builder.compile()
print(open_graph.get_graph().draw_mermaid())

In [ ]:
for update in open_graph.stream(dispatch_input(), config={"recursion_limit": 8},
                                stream_mode="updates"):
    for node, delta in update.items():
        last = delta["messages"][-1]
        calls = [c["name"] for c in getattr(last, "tool_calls", [])]
        print(f"{node}: {calls or last.content[:48]}")

print("outbox:", OUTBOX)

**It sent. Nobody asked.**

- the model decided, the loop executed, `OUTBOX` grew by one
- a wrong guess by the loop is a wrong email out the door
- one more look first: the same run in `messages` mode, the chat-UI feed

In [ ]:
from langchain_core.messages import ToolMessage

for chunk, meta in open_graph.stream(dispatch_input(), config={"recursion_limit": 8},
                                     stream_mode="messages"):
    if isinstance(chunk, ToolMessage):  # tool results interleave with the tokens
        print(f"[{chunk.name}] {chunk.content}")
    elif chunk.content:
        print(chunk.content, end="", flush=True)

print("\noutbox size:", len(OUTBOX))

## The human gate

**"Send only what a human meant" is not a regex. It is a pause.**

- intent needs a human, so the tool must stop and ask one
- `interrupt()` pauses the run anywhere, a `@tool` body included
- a pause, not an exception: nothing raises, nothing unwinds

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, interrupt

plain_send = send_email  # keep the ungated tool; middleware wants it later


@tool
def send_email(to: str, body: str) -> str:
    """Send an email to an address from lookup_contact. A human approves each send."""
    print("send_email: entered")  # watch this line; it matters on resume
    decision = interrupt({"to": to, "body": body})  # parks the run right here
    if not decision["approved"]:
        return f"not sent: {decision.get('reason', 'refused')}"
    OUTBOX.append({"to": to, "body": body})  # the side effect sits after the gate
    return f"sent to {to}"


OUTBOX.clear()  # a clean mailbox for the gated story
gated_builder = dispatch_builder([lookup_contact, send_email])
gated = gated_builder.compile(checkpointer=InMemorySaver())  # SqliteSaver drops in too
mail_cfg = {"configurable": {"thread_id": "mail"}, "recursion_limit": 8}

In [ ]:
paused = gated.invoke(dispatch_input(), config=mail_cfg)

pause = paused["__interrupt__"][0]  # a new key next to the usual state
print("payload:", pause.value)
print("id:     ", pause.id)
print("outbox: ", OUTBOX)  # empty: the send never happened

snapshot = gated.get_state(mail_cfg)
print("next:   ", snapshot.next)  # the node that will resume
print("pending:", snapshot.tasks[0].interrupts)

**Parked, not crashed.**

- the payload is yours to design: whatever the approver needs to decide
- the run sits in the checkpointer, not on the Python call stack
- with session 4's `SqliteSaver` in place of `InMemorySaver`, the kernel could die right now and any process holding the `thread_id` could pick the run up
- several pending interrupts? resume takes a dict keyed by interrupt id (mention only)

In [ ]:
approved = gated.invoke(Command(resume={"approved": True}), config=mail_cfg)

print([m.content for m in approved["messages"] if m.type == "tool"])
print(approved["messages"][-1].content)
print("outbox:", OUTBOX)  # exactly one send, the one a human approved

**The resume value is `interrupt()`'s return. And the tool ran twice.**

- `send_email: entered` printed on the pause and again on the resume
- on resume the interrupted tool re-executes from its first line
- that is why the `OUTBOX` append lives after the gate, never before

In [ ]:
bare = gated_builder.compile()  # same graph, checkpointer forgotten

looks_fine = bare.invoke(dispatch_input(), config={"recursion_limit": 8})
print("paused?", "__interrupt__" in looks_fine)  # the pause works; nothing warns

try:
    bare.invoke(Command(resume={"approved": True}), config={"recursion_limit": 8})
except RuntimeError as error:
    print("RuntimeError:", error)  # parked nowhere, so it cannot come back

**The silent trap of the session: pausing is free, resuming is not.**

- the pause returns `__interrupt__` as if all were well
- resume needs state that survived; there is none, and the run is lost
- every HITL graph compiles with a checkpointer, full stop

**How a client sees the pause, mid-stream.**

- in `updates` mode the interrupt arrives as its own `{"__interrupt__": ...}` chunk
- in `values` mode it rides appended to the final state chunk
- resume through `stream` too: the same run, finished in a second stream

In [ ]:
client_cfg = {"configurable": {"thread_id": "mail-client"}, "recursion_limit": 8}

for update in gated.stream(dispatch_input(), config=client_cfg, stream_mode="updates"):
    for node, payload in update.items():
        if node == "__interrupt__":  # the pause arrives as a chunk mid-stream
            print("needs approval:", payload[0].value["to"])
        else:
            print(node, "ran")

decision = {"approved": True}  # a console client would ask a human here, via input()
for update in gated.stream(Command(resume=decision), config=client_cfg,
                           stream_mode="updates"):
    print("resumed:", *update)

## Breakpoints, and the pre-1.0 world

**`interrupt_before=["tools"]` stops before the node. A debugger, not a gate.**

- set at compile time, or per invoke; continue with `invoke(None)`
- step through a run while `get_state().next` is non-empty
- tutorials teaching `NodeInterrupt` or `interrupt_before` as the approval flow describe the pre-1.0 world
- current docs file `interrupt_before` under debugging; `NodeInterrupt` is gone from them entirely, though it still imports on 1.2.9 — exactly how those old tutorials keep running

In [ ]:
dbg = open_builder.compile(checkpointer=InMemorySaver(), interrupt_before=["tools"])
dbg_cfg = {"configurable": {"thread_id": "debug"}, "recursion_limit": 8}

dbg.invoke(dispatch_input(), config=dbg_cfg)
while dbg.get_state(dbg_cfg).next:  # something is still pending
    print("stopped before:", dbg.get_state(dbg_cfg).next)
    dbg.invoke(None, config=dbg_cfg)  # None: continue, do not add input

print("outbox size:", len(OUTBOX))  # the ungated graph sent without asking again

## The same gate, from config

**The session-3 `create_agent` assistant gets HITL without tool surgery.**

- `HumanInTheLoopMiddleware` wraps the ungated tool
- `interrupt_on` names the tools that must wait for a human
- same pause, same resume, a richer payload

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware

assistant = create_agent(
    model,
    tools=[lookup_contact, plain_send],  # ungated tool: the gate now lives in config
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"send_email": True})],
    checkpointer=InMemorySaver(),
    system_prompt=SYSTEM,
)
assist_cfg = {"configurable": {"thread_id": "assist"}, "recursion_limit": 8}

mw_paused = assistant.invoke({"messages": [{"role": "user", "content": REQUEST}]},
                             config=assist_cfg)
mw_pause = mw_paused["__interrupt__"][0]
print("action_requests:", mw_pause.value["action_requests"])
print("review_configs: ", mw_pause.value["review_configs"])

In [ ]:
mw_done = assistant.invoke(Command(resume={"decisions": [{"type": "approve"}]}),
                           config=assist_cfg)

print(mw_done["messages"][-1].content)
print("outbox size:", len(OUTBOX))  # the middleware ran the real tool after the yes

**Four decisions, richer than a yes or no.**

- `approve`, `edit` (patch the args), `reject`, `respond` (answer without running)
- `interrupt_on` values: `True`, `False`, or an `InterruptOnConfig`
- `InterruptOnConfig(when=...)` gates on arguments: externals only, after hours only
- Agent Chat UI renders `action_requests` as approval buttons, for free

## Time travel

**Every step of the `mail` thread is a checkpoint, and it is still there.**

- `get_state_history` walks them, newest first
- each snapshot: step, pending nodes, values, and a `checkpoint_id` address

In [ ]:
history = list(gated.get_state_history(mail_cfg))  # newest first

for snap in history:
    print(f"step {snap.metadata['step']:2} | next {str(snap.next):12}",
          f"| {len(snap.values.get('messages', []))} messages",
          f"| id ..{snap.config['configurable']['checkpoint_id'][-8:]}",
          "| pending interrupt" if snap.tasks and snap.tasks[0].interrupts else "")

**A config with a `checkpoint_id` addresses one moment.**

- `invoke(None, config=that_config)` replays from the moment, as a new branch
- `update_state(past_config, {...})` forks with edited state and returns the new address (mention; no demo today)
- the thread keeps both futures; nothing is overwritten

In [ ]:
pre_send = next(s for s in history  # the moment before the send, gate still pending
                if s.next == ("tools",) and s.tasks and s.tasks[0].interrupts)
print("replay from:", pre_send.config["configurable"]["checkpoint_id"][-8:])

forked = gated.invoke(None, config={**pre_send.config, "recursion_limit": 8})
print("asked again:", forked["__interrupt__"][0].value["to"])  # the gate re-fired

rejected = gated.invoke(Command(resume={"approved": False, "reason": "tone too casual"}),
                        config=mail_cfg)  # the thread tip is now the fork
print("future one:", [m.content for m in approved["messages"] if m.type == "tool"][-1])
print("future two:", [m.content for m in rejected["messages"] if m.type == "tool"][-1])
print(rejected["messages"][-1].content)
print("outbox size:", len(OUTBOX))  # the refused branch added nothing

**One approval, two traces: the pause run and the resume run.**

- each `invoke` is its own trace; the gate spans two of them
- the approved send shows in the resume trace's `tools` span
- either trace exports; the pair tells the whole story

In [ ]:
import os

from langfuse import get_client
from langfuse.langchain import CallbackHandler

client = get_client()  # reads LANGFUSE_HOST and both keys from the environment
print("server:", os.getenv("LANGFUSE_HOST"), "| up:", client.auth_check())

handler = CallbackHandler()  # a fresh handler per run gives one trace per run
traced_cfg = {"configurable": {"thread_id": "mail-traced"},
              "recursion_limit": 8, "callbacks": [handler]}

gated.invoke(dispatch_input(), config=traced_cfg)  # trace one: up to the pause
done = gated.invoke(Command(resume={"approved": True}), config=traced_cfg)  # trace two

client.flush()  # a notebook kernel never exits, so nothing sends without this
print(done["messages"][-1].content)

## Agent Chat UI

**A ready chat front end for any local graph. Pointer only, nothing to install today.**

- github.com/langchain-ai/agent-chat-ui, hosted at agentchat.vercel.app
- point it at a local `langgraph dev` server: it renders tool calls, interrupts as approval buttons, and time travel
- try it at home; the course deploys its own stack in session 8

## Practice

**Gate the one most dangerous tool of your own agent, in your own repository.**

1. pick the tool whose wrong call hurts most: delete, send, pay
2. on `create_agent`: `HumanInTheLoopMiddleware`; on a hand-built graph: `interrupt()` inside the tool
3. compile with a checkpointer; reproduce the trap cell's failure once to believe it
4. side effects after the gate: your tool re-runs on resume

**Then build the console client this notebook deliberately avoided.**

5. stream with `stream_mode=["messages", "updates"]`: print tokens from the `("messages", ...)` tuples
6. an `("updates", {"__interrupt__": ...})` tuple is the pause: ask via `input()`, resume with `Command(resume=...)`
7. `input()` belongs in your client script, never in a notebook cell
8. fork a real dialogue from a past checkpoint and keep both continuations

**Required artifact: `runs/session-06.md`, committed.**

- a streamed transcript with one refused and one approved dangerous call
- two continuations of one dialogue, forked from the same checkpoint
- the exported trace where the approved call is visible: a gated run leaves a pause trace and a resume trace, either counts
- project: the confirmation gate is cheap and counts as a defense-day security measure
- project: streaming visibly upgrades the demo; session 8 wires it to a bot

**Stretch, if you finish early.**

- handle an `edit` decision: the human fixes the recipient before the send

## Next time

**Full observability: cost per node, sessions and users in traces, prompt management.**